# Exercise 3.1: Aggregating and Summarizing (Angola trade)

Angola's international trade of goods, published by INE. Twenty two years of
exports and imports, partner by partner, in thousands of US dollars.

You will practice: `groupby`, `.agg`, grouping by two variables, `stack` and
`unstack`, pivot tables, cross tabulations, and filtering groups.

**PT:** O comercio internacional de bens de Angola, publicado pelo INE. Vinte e
dois anos de exportacoes e importacoes, parceiro a parceiro, em milhares de
dolares.

Vai praticar: `groupby`, `.agg`, agrupar por duas variaveis, `stack` e `unstack`,
tabelas dinamicas, tabelas de contingencia, e filtrar grupos.

> **Pipeline:** reads `0_raw/angola/international_trade`, writes `20_processed/`.

### Path Setup (run first)

**PT:** Configuracao dos caminhos.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_RAW_DIR = '../../data/0_raw/angola'
DATA_PROC_DIR = '../../data/20_processed'

TRADE_DIR = 'international_trade'
PARTNERS_FILE = 'Comercio Externo de Bens por Países Parceiros.xlsx'

trade_dir = os.path.join(DATA_RAW_DIR, TRADE_DIR)
partners_path = os.path.join(trade_dir, PARTNERS_FILE)

pd.set_option('display.float_format', lambda x: f'{x:,.1f}')
print('File:', partners_path)
print('Exists?:', os.path.exists(partners_path))

---

## Task 1: Load both flows into one table

The workbook keeps exports and imports on separate sheets with the same shape.
Load both, tidy the column names, and stack them into one table with a `flow`
column saying which is which.

**What to do:** load each sheet with `skiprows=2`, convert the names to snake
case, keep only the rows where the country name is filled in, add `flow`, then
concatenate.

**PT:** O ficheiro tem exportacoes e importacoes em folhas separadas com a mesma
forma. Carregue as duas, arrume os nomes, e empilhe numa so tabela com uma coluna
`flow`.

**O que fazer:** carregue cada folha com `skiprows=2`, converta os nomes para
snake case, mantenha as linhas com o nome do pais preenchido, adicione `flow`, e
concatene.

In [ ]:
def to_snake_case(columns):
    """Lower case, strip accents, join words with underscores.

    Minusculas, sem acentos, palavras unidas por underscore.
    """
    return (columns
            .str.replace('\n', ' ', regex=False)
            .str.strip()
            .str.lower()
            .str.normalize('NFKD')
            .str.encode('ascii', errors='ignore')
            .str.decode('utf-8')
            .str.replace(' ', '_', regex=False))


sheets = {'Exportação por Países (USD)': 'Export',
          'Importação por Países (USD)': 'Import'}

frames = []
for sheet_name, flow in sheets.items():
    sheet = pd.read_excel(partners_path, sheet_name=sheet_name, skiprows=2)
    sheet.columns = to_snake_case(sheet.columns)
    sheet =   # your code here: keep the rows where pais is filled in
    sheet['flow'] =   # your code here: label this sheet's flow
    frames.append(sheet)

wide = pd.concat(frames, ignore_index=True)
print('wide:', wide.shape)
print(wide['flow'].value_counts())

**Questions:**

- How many rows did you load, and how many per flow?
- Where does the year live at this point? Can you group by it yet?
- Why add `flow` before the concat rather than after?

**PT:** Quantas linhas carregou e quantas por fluxo? Onde esta o ano nesta fase, e
ja consegue agrupar por ele? Porque adicionar `flow` antes do concat?

---

## Task 2: Turn the year columns into rows with `stack()`

`groupby` needs the thing you group by to be a column, not a column *name*. Right
now the year lives in the header, so there is nothing to group.

`stack()` pushes columns down into the index, making the table longer and
narrower. It is the exact opposite of `unstack()`, which you will use in Task 5.

**What to do:** set the identifying columns as the index, keep only the year
columns, `stack()` them, reset the index, name the columns, and turn the year
text into a number.

**PT:** O `groupby` precisa que aquilo que agrupa seja uma coluna, e nao o nome
de uma coluna. Agora o ano esta no cabecalho, por isso nao ha nada para agrupar.

`stack()` empurra as colunas para o indice, tornando a tabela mais comprida e
estreita. E o oposto de `unstack()`, que vai usar na Tarefa 5.

**O que fazer:** ponha as colunas de identificacao no indice, fique so com as
colunas de ano, faca `stack()`, reponha o indice, dê nome as colunas, e converta
o texto do ano em numero.

In [ ]:
year_columns = [col for col in wide.columns if col.startswith('ano_')]
print('Year columns:', len(year_columns), year_columns[:3], '...', year_columns[-1:])

trade = (wide
         # your code here: set_index on the three id columns, keep the year
         # columns, stack(), then reset_index()
         # o seu codigo aqui: set_index, stack(), reset_index()
         )
trade.columns = ['country_code', 'country_name', 'flow', 'year', 'value_thousand_usd']

# The year arrives as the text 'ano_2004' / O ano chega como o texto 'ano_2004'
trade['year'] = trade['year'].str.replace('ano_', '', regex=False).astype(int)

print('long:', trade.shape)
trade.head()

**Questions:**

- How many rows does the long table have? Check the arithmetic against partners,
  years and flows.
- What is one row now?
- In one sentence each, what do `stack()` and `unstack()` do?

**PT:** Quantas linhas tem a tabela longa? Confirme a aritmetica. O que e agora
uma linha? Numa frase, o que fazem `stack()` e `unstack()`?

---

## Task 3: First groupings

`groupby` splits the table into groups, applies a function to each, and puts the
results back together.

**What to do:** total the value by flow, then by year, and count the rows per
flow with `size()`.

**PT:** O `groupby` divide a tabela em grupos, aplica uma funcao a cada um, e
junta os resultados.

**O que fazer:** some o valor por fluxo, depois por ano, e conte as linhas por
fluxo com `size()`.

In [ ]:
print('Total by flow / Total por fluxo (thousand USD):')
print(trade.  # your code here: group by flow and sum the value )

In [ ]:
print('Rows per flow / Linhas por fluxo:')
print(trade.  # your code here: group by flow and use size() )

In [ ]:
by_year =   # your code here: total the value by year
print(by_year.tail(6))

**Questions:**

- Which flow is larger over the whole period, and by how much?
- What does `size()` count, and how does it differ from `sum()`?
- Look at trade by year. Is it a smooth trend?

**PT:** Que fluxo e maior no periodo e por quanto? O que conta `size()` e em que
difere de `sum()`? O comercio por ano e uma tendencia suave?

---

## Task 4: Several statistics at once with `.agg()`

One number per group rarely tells the whole story. `.agg()` gives you several in
one pass, and named aggregations make the output readable.

**What to do:** for exports only, compute the mean, median, standard deviation
and count by year, then repeat with named aggregations.

**PT:** Um so numero por grupo raramente conta a historia toda. `.agg()` da
varios de uma vez, e nomear as agregacoes torna o resultado legivel.

**O que fazer:** so para as exportacoes, calcule media, mediana, desvio padrao e
contagem por ano, e depois repita com agregacoes nomeadas.

In [ ]:
exports = trade[trade['flow'] == 'Export']

exports.groupby('year')['value_thousand_usd'].  # your code here: agg with a list
                                               # of mean, median, std, count

In [ ]:
exports.groupby('year')['value_thousand_usd'].agg(
    # your code here: name them average, middle, spread, partners
    # o seu codigo aqui: agregacoes nomeadas
).tail(5)

**Questions:**

- For 2025, compare the mean and the median export per partner. How large is the
  gap?
- Is that gap a data error? What does it tell you about Angola's trade?
- Which of the two would you publish, and what would you say alongside it?

**PT:** Em 2025, compare a media e a mediana por parceiro. A diferenca e um erro?
O que diz sobre o comercio de Angola? Qual publicaria?

---

## Task 5: Group by two variables, then `unstack()`

Grouping by two columns gives a result with two index levels, which reads poorly.
`unstack()` moves the innermost level up into the columns and turns it into a
table.

**What to do:** total the value by year and flow, look at the stacked result,
then `unstack()` it into a year by flow table.

**PT:** Agrupar por duas colunas da um resultado com dois niveis de indice, que
se le mal. `unstack()` move o nivel de dentro para as colunas.

**O que fazer:** some o valor por ano e fluxo, veja o resultado empilhado, e
depois faca `unstack()` para obter uma tabela de ano por fluxo.

In [ ]:
by_year_flow =   # your code here: total the value by year and flow
print(by_year_flow.tail(6))

In [ ]:
table =   # your code here: unstack the flow level into columns
table.tail(6)

In [ ]:
# A balance column falls out once the flows are side by side
# Com os fluxos lado a lado, a balanca comercial sai naturalmente
table['balance'] =   # your code here: exports minus imports
table.tail(6)

**Questions:**

- How many rows does the grouped result have before `unstack()`, and what shape
  after?
- What were exports and imports in 2025, and what is the balance?
- Has the surplus widened or narrowed in recent years?

**PT:** Quantas linhas tem o resultado antes do `unstack()` e que forma tem
depois? Quais foram as exportacoes e importacoes em 2025? O excedente alargou ou
estreitou?

---

## Task 6: Pivot tables and margins

`pivot_table` does the grouping and the reshaping in one call, and `margins=True`
adds the totals row and column.

Careful: margins use the same `aggfunc`, so with `mean` they are averages and not
sums.

**What to do:** build a pivot table of the value by year and flow for the last
three years, with totals.

**PT:** O `pivot_table` faz o agrupamento e a remodelacao numa so chamada, e
`margins=True` acrescenta a linha e a coluna de totais, calculadas com a mesma
funcao.

**O que fazer:** construa uma tabela dinamica do valor por ano e fluxo para os
ultimos tres anos, com totais.

In [ ]:
recent = trade[trade['year'] >= 2023]

pd.pivot_table(
    # your code here: values, index=year, columns=flow, aggfunc='sum',
    # margins=True, margins_name='Total'
    # o seu codigo aqui
)

**Questions:**

- What does the bottom right cell of the table mean?
- If you changed `aggfunc` to `'mean'`, would the `Total` row still be the sum of
  the rows above it?
- When would you use `pivot` instead of `pivot_table`?

**PT:** O que significa a celula inferior direita? Com `aggfunc='mean'`, a linha
`Total` continuaria a ser a soma? Quando usaria `pivot` em vez de `pivot_table`?

---

## Task 7: Cross tabulation of two categories

`crosstab` counts how often each combination of two categorical variables occurs.
Value is a number, not a category, so first turn it into a size band.

**What to do:** write `size_band` returning `'None'` for zero, `'Under 100'`,
`'100 to 10k'` and `'Over 10k'`, apply it, then cross tabulate `flow` against the
band, first as counts and then as row percentages.

**PT:** O `crosstab` conta com que frequencia ocorre cada combinacao de duas
variaveis categoricas. O valor e um numero, por isso primeiro converta-o numa
faixa.

**O que fazer:** escreva `size_band` que devolve `'None'` para zero, `'Under
100'`, `'100 to 10k'` e `'Over 10k'`, aplique, e faca a tabela cruzada de `flow`
contra a faixa, em contagens e depois em percentagens por linha.

In [ ]:
def size_band(value):
    """Band one trade value / Classificar um valor de comercio."""
    # your code here: 'None', 'Under 100', '100 to 10k', 'Over 10k'
    # o seu codigo aqui
    return


trade['size_band'] =   # your code here: apply size_band to the value column
pd.crosstab(  # your code here: flow against size_band )

In [ ]:
# Row percentages: each row sums to 100
# Percentagens por linha: cada linha soma 100
(pd.crosstab(  # your code here: add normalize='index' ) * 100).round(1)

**Questions:**

- What share of export records is `None`, and what share of import records? What
  does that difference say?
- Look at the `Over 10k` column as well. Does it tell the same story?
- What is the difference between `normalize='index'`, `'columns'` and `'all'`?

**PT:** Que percentagem dos registos de exportacao e `None`, e de importacao? O
que diz essa diferenca? E a coluna `Over 10k`? Qual e a diferenca entre os tres
modos de `normalize`?

---

## Task 8: Chain operations to answer a question

Aggregations chain. Group, then sort, then take the head, all in one expression.

**What to do:** find the ten partners with the largest total exports over the
whole period, then show mean, median and count per partner sorted by median.

**PT:** As agregacoes encadeiam-se: agrupar, ordenar, e ficar com as primeiras
linhas numa so expressao.

**O que fazer:** encontre os dez parceiros com maiores exportacoes totais no
periodo, e depois mostre media, mediana e contagem por parceiro ordenadas pela
mediana.

In [ ]:
top_partners = (exports
                # your code here: group by country_name, sum, sort
                # descending, take the first 10
                )
top_partners

In [ ]:
(exports
 .groupby('country_name')['value_thousand_usd']
 .agg(['mean', 'median', 'count'])
 .  # your code here: sort by median descending, then head(10)
 )

**Questions:**

- Which partner is largest by total exports, and by how much over the second?
- Does sorting by median give the same top ten as sorting by total? Why not?
- Which ranking answers "who matters most to the economy"?

**PT:** Qual e o maior parceiro por exportacoes totais e por quanto face ao
segundo? Ordenar pela mediana da a mesma lista? Qual responde a "quem pesa mais
na economia"?

---

## Task 9: Filter on a property of the group

Sometimes the filter depends on the group and not on the row: keep only partners
that reach a certain size. That takes two steps, compute the group statistic,
then select the rows whose group qualifies.

**What to do:** total the trade per partner, keep the partners above 1,000,000
thousand USD, and filter the table down to them with `isin`.

**PT:** Por vezes o filtro depende do grupo e nao da linha. Sao dois passos:
calcular a estatistica do grupo e depois selecionar as linhas cujo grupo passa.

**O que fazer:** some o comercio por parceiro, fique com os que passam de
1.000.000 milhares de dolares, e filtre a tabela com `isin`.

In [ ]:
per_partner =   # your code here: total the trade per partner
major =   # your code here: the names whose total passes 1,000,000

print('Partners kept / Parceiros mantidos:', len(major), 'of', trade['country_name'].nunique())

trade_major =   # your code here: keep only the rows of those partners
print('Rows:', len(trade), '->', len(trade_major))
print('Share of all trade kept: '
      f"{trade_major['value_thousand_usd'].sum() / trade['value_thousand_usd'].sum() * 100:.1f}%")

**Questions:**

- How many partners pass the threshold, and what share of total trade do they
  carry?
- Does that agree with what the crosstab suggested in Task 7?
- The threshold was chosen, not discovered. Where should it be recorded?

**PT:** Quantos parceiros passam o limiar e que percentagem do comercio
representam? Concorda com a Tarefa 7? Onde deve ser registado o limiar?

---

## Task 10: Save

**What to do:** write the long table and the year by flow summary to
`20_processed/` with `index=False` for the long one.

**PT:** **O que fazer:** grave a tabela longa e o resumo por ano e fluxo em
`20_processed/`.

In [ ]:
os.makedirs(DATA_PROC_DIR, exist_ok=True)

long_out = os.path.join(DATA_PROC_DIR, 'angola_trade_long.csv')
summary_out = os.path.join(DATA_PROC_DIR, 'angola_trade_year_flow.csv')

# your code here: write trade with index=False, and table with its index
# o seu codigo aqui

print('long:', trade.shape, '| summary:', table.shape)

**Questions:**

- What is the difference between the two files you wrote?
- Why does one keep its index and the other not?

**PT:** Qual e a diferenca entre os dois ficheiros? Porque um mantem o indice e o
outro nao?